# Noema public discovery analysis — 2026-08-27

Decision: choose the next growth intervention without guessing from content volume. The notebook checks the live public surface, then combines it with a bounded public-search observation. Authenticated reader analytics and Search Console data are intentionally not inferred.

In [1]:
from html.parser import HTMLParser
from pathlib import Path
from urllib.request import Request, urlopen
from xml.etree import ElementTree
import json
import re
import sqlite3

ORIGIN = 'https://noema-learn.uk'
SITEMAP = f'{ORIGIN}/sitemap.xml'
OBSERVED_AT = '2026-08-27'

def fetch_text(url):
    request = Request(url, headers={'User-Agent': 'Noema growth analysis/2026-08-27'})
    with urlopen(request, timeout=20) as response:
        return response.status, response.geturl(), response.read().decode('utf-8', errors='replace')


In [2]:
class PublicPageParser(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.title = []
        self.h1 = []
        self.meta = []
        self.links = []
        self.text = []
        self.json_ld = []
        self._title = False
        self._h1 = False
        self._skip = 0
        self._json_ld = False

    def handle_starttag(self, tag, attrs):
        values = dict(attrs)
        if tag == 'title': self._title = True
        if tag == 'h1': self._h1 = True
        if tag in {'script', 'style', 'noscript', 'svg', 'template'}: self._skip += 1
        if tag == 'script' and values.get('type', '').lower() == 'application/ld+json': self._json_ld = True
        if tag == 'meta': self.meta.append(values)
        if tag == 'link': self.links.append(values)

    def handle_endtag(self, tag):
        if tag == 'title': self._title = False
        if tag == 'h1': self._h1 = False
        if tag == 'script': self._json_ld = False
        if tag in {'script', 'style', 'noscript', 'svg', 'template'} and self._skip: self._skip -= 1

    def handle_data(self, data):
        cleaned = re.sub(r'\s+', ' ', data).strip()
        if self._title and cleaned: self.title.append(cleaned)
        if self._h1 and cleaned: self.h1.append(cleaned)
        if self._json_ld: self.json_ld.append(data)
        if not self._skip and cleaned: self.text.append(cleaned)

def inspect_page(url):
    status, final_url, html = fetch_text(url)
    parser = PublicPageParser()
    parser.feed(html)
    meta = {(item.get('name') or item.get('property') or '').lower(): item.get('content', '') for item in parser.meta}
    canonical = next((item.get('href', '') for item in parser.links if 'canonical' in item.get('rel', '').lower().split()), '')
    schema_text = ''.join(parser.json_ld)
    return {
        'url': url, 'status': status, 'final_url': final_url,
        'title': ' '.join(parser.title), 'description': meta.get('description', ''),
        'canonical': canonical, 'robots': meta.get('robots', ''),
        'h1_count': 1 if parser.h1 else 0,
        'article_schema': ('Article' in schema_text or 'BlogPosting' in schema_text),
        'visible_characters': sum(len(part) for part in parser.text)
    }


In [3]:
_, _, sitemap_xml = fetch_text(SITEMAP)
root = ElementTree.fromstring(sitemap_xml)
namespace = '{http://www.sitemaps.org/schemas/sitemap/0.9}'
sitemap_urls = [node.text for node in root.findall(f'{namespace}url/{namespace}loc') if node.text]
pages = [inspect_page(url) for url in sitemap_urls]
article_pages = [page for page in pages if '/articles/' in page['url'] and not page['url'].endswith('/articles/')]

def is_blocked(page):
    return (
        page['status'] != 200 or page['final_url'].rstrip('/') != page['url'].rstrip('/') or
        not page['title'] or not page['description'] or not page['canonical'] or
        'noindex' in page['robots'].lower() or page['h1_count'] != 1 or
        ('/articles/' in page['url'] and not page['article_schema'])
    )

technical_blockers = [page for page in pages if is_blocked(page)]
public_summary = {
    'observed_at': OBSERVED_AT,
    'sitemap_urls': len(sitemap_urls),
    'article_pages': len(article_pages),
    'technical_blockers': len(technical_blockers),
    'shortest_article_visible_characters': min(page['visible_characters'] for page in article_pages),
    'longest_article_visible_characters': max(page['visible_characters'] for page in article_pages),
}
surface_rows = [
    ('記事', len(article_pages)),
    ('シリーズ', sum('/series/' in url and not url.endswith('/series/') for url in sitemap_urls)),
    ('テーマ', sum('/topics/' in url for url in sitemap_urls)),
]
surface_rows.append(('その他', len(sitemap_urls) - sum(row[1] for row in surface_rows)))
database = sqlite3.connect(':memory:')
database.execute('CREATE TABLE public_summary (observed_at TEXT, sitemap_urls INTEGER, article_pages INTEGER, technical_blockers INTEGER, shortest_article_visible_characters INTEGER, longest_article_visible_characters INTEGER)')
database.execute('INSERT INTO public_summary VALUES (?, ?, ?, ?, ?, ?)', tuple(public_summary.values()))
database.execute('CREATE TABLE public_surface (surface TEXT, url_count INTEGER)')
database.executemany('INSERT INTO public_surface VALUES (?, ?)', surface_rows)
PUBLIC_SUMMARY_SQL = "SELECT sitemap_urls, article_pages, technical_blockers, shortest_article_visible_characters, longest_article_visible_characters FROM public_summary WHERE observed_at = '2026-08-27'"
PUBLIC_SURFACE_SQL = 'SELECT surface, url_count FROM public_surface ORDER BY url_count DESC, surface ASC'
public_summary_query_result = database.execute(PUBLIC_SUMMARY_SQL).fetchall()
public_surface_query_result = database.execute(PUBLIC_SURFACE_SQL).fetchall()
{'summary': public_summary_query_result, 'surface': public_surface_query_result}


{'summary': [(27, 16, 0, 5551, 11699)],
 'surface': [('記事', 16), ('その他', 6), ('テーマ', 3), ('シリーズ', 2)]}

In [4]:
NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / 'public-search-observations.json').exists() else Path('docs/growth/2026-08-27')
search_observations = json.loads((NOTEBOOK_DIR / 'public-search-observations.json').read_text())
search_summary = {
    'queries_observed': len(search_observations['queries']),
    'queries_with_noema_article_result': sum(item['noemaArticleResultObserved'] for item in search_observations['queries']),
    'is_exhaustive_index_coverage': False,
}
decision_evidence = [
    ('Live public crawl', '27 sitemap URL, 16 articles, 0 blockers', 'Technical remediation is not the next bottleneck.', 'Public HTTP and markup only.'),
    ('Public search sample', '0 Noema article results in 8 queries', 'Prioritize discovery notification before more content volume.', 'Not exhaustive index coverage.'),
    ('Authenticated behavior data', 'Unavailable', 'Do not rank article topics or claim traffic uplift yet.', 'Studio and Search Console sign-in required.'),
]
database.execute('CREATE TABLE decision_evidence (evidence TEXT, observed TEXT, decision_use TEXT, limitation TEXT)')
database.executemany('INSERT INTO decision_evidence VALUES (?, ?, ?, ?)', decision_evidence)
DECISION_EVIDENCE_SQL = 'SELECT evidence, observed, decision_use, limitation FROM decision_evidence ORDER BY evidence ASC'
decision_evidence_query_result = database.execute(DECISION_EVIDENCE_SQL).fetchall()
{'search': search_summary, 'decision_evidence': decision_evidence_query_result}


{'search': {'queries_observed': 8,
  'queries_with_noema_article_result': 0,
  'is_exhaustive_index_coverage': False},
 'decision_evidence': [('Authenticated behavior data',
   'Unavailable',
   'Do not rank article topics or claim traffic uplift yet.',
   'Studio and Search Console sign-in required.'),
  ('Live public crawl',
   '27 sitemap URL, 16 articles, 0 blockers',
   'Technical remediation is not the next bottleneck.',
   'Public HTTP and markup only.'),
  ('Public search sample',
   '0 Noema article results in 8 queries',
   'Prioritize discovery notification before more content volume.',
   'Not exhaustive index coverage.')]}

## Data quality and decision

- The public crawl is current, reproducible, and covers every URL declared in the live sitemap.
- The search sample is directional only; it is not an engine-wide index count.
- Authenticated reader analytics and Google Search Console were unavailable, so no traffic, conversion, trend, or causal uplift claim is made.
- With no public technical blocker observed and no article result observed in the bounded search sample, the next reversible intervention is automated discovery notification via IndexNow. This complements the sitemap for participating engines; it does not replace Search Console or make a Google-indexing claim.